### Question answering with LLM on a data that it has never seen 

There are three ways, if a model is not trained on the data and you like to answer the question from the model on those data:
1. Fine-tuning
2. Prompt engineering (without retrieval of information)
3. Retrieval Augmented Generation

#### 1. Fine-tuning

The cons of fine-tuning:
- It can be expensive.
- Number of parameters of the model may not be sufficient to capture all the knowledge we want to teach to it.
- Fine tuning is not additive.

The pros of fine-tuning:

- Higher quality resukts compared to prompt engineering!
- Don't need to provide context on the neweer data as the model already knows about it. You can directly ask the question. 

#### 2. Prompt engineering

We can compensate the fine-tuning with the prompt engineering few-shot prompting!

Few-shot prompting is a technique where you provide an AI model with some completed examples (or "shots") within your prompt. By observing these examples, the AI learns the desired pattern, tone, and output format before tackling your actual query.

Cons: You need a bigger contect >> More tokes >> Computationally expensive!

#### 3. Retrieval Augmented Generation (RAG)

We create chunk from documents.
Then we create embeddings of these sentences such that each embedding is a vector of a fixed-size that captures the meaning of each sentence.
Then we store all of these embeddings into a vector database.

Then we also take the query, which is a sentence. We convert it into an embedidng using the same model that we used to convert the documents into embeddings. We search this query embedding into the database which has all the document embeddings each representing a sentence from our document. It comes up with some results, with the best matching embeddings for our particular query!

Then we set the context with the result, use it in prompt template to set the context, and answering the questions based on this document.

_Note:_ In RAG, We are using prompt engineering and inroducing a databse called vector database to access the context given our query! So, it can retireve the context necessary to answer our particular question, feed it to the language model and then the language model using the context and our question will be able to answer!

_Note:_ For real-time querying of unseen data, RAG is usually the best approach because it prevents the model from hallucinating and ensures it references up-to-date facts.

Note: Instead of converting the generated result to a text, can we directly pass the generated embedding as context? An ML paper on this can be written: Mashhood








### Full RAG Pipeline

Complete Retrieval-Augmented Generation with LangChain Expresiion Language (LCEL).

In [2]:
# Create a document containing information about channel and its creators
%mkdir -p data
with open('data/breakdown.txt', 'w') as f:
    f.write("BreakDown is an educational YouTube channel run by AI researchers Mashhood Raza and Aamna Hasan. \n" +
             "They teach machine learning and deep learning models on their YouTube channel.\n" +
            "They're both passionate about AI and it's applications, and Aamna is very fast in learning new concepts.\n" +
            "They have just started publishing videos. To date, the channel has released three videos.\n" +
            "Apart from machine learning, they like walkiing together at sunset and travelling.\n")

#### Load → Split → Embed → Retrieve → Prompt → RAG Chain

#### Step 1 — Load documents
LangChain provides loaders for dozens of file formats. 
They all return a list of Document objects with metadata and page_content.

In [63]:
# Import module for loading text files 
from langchain_community.document_loaders import TextLoader

# Loads the text file and creates a list of documents.
loader= TextLoader("./data/breakdown.txt") #each document has metadata (source, page number, etc.) and page_content (the text).
docs = loader.load() # Load the documents from the text file

print(len(docs)) # Display the number of documents loaded
print(docs[0].metadata) # Display the metadata of the first document
print(docs[0].page_content[:200]) # Display the first 200 characters of the first document

1
{'source': './data/breakdown.txt'}
BreakDown is an educational YouTube channel run by AI researchers Mashhood Raza and Aamna Hasan. 
They teach machine learning and deep learning models on their YouTube channel.
They're both passionate


#### Step 2 — Chunk the text
LLMs have context limits, and embeddings work best on focused passages — not entire documents. You split documents into smaller chunks before embedding. There are different chunking strategies that can be used.

In [64]:
# Import module for splitting the documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Splits the documents into smaller chunks based on the specified chunk size and overlap.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,       # characters per chunk
    chunk_overlap=20,     # overlap to preserve context across chunk boundaries
)
chunks = splitter.split_documents(docs) 

print(len(chunks)) # Display the total number of chunks
print(chunks[0].metadata) # Display the metadata of the first chunk
print(chunks[0].page_content[:50]) # Display the first 50 characters of the first chunk


6
{'source': './data/breakdown.txt'}
BreakDown is an educational YouTube channel run by


#### Step 3 — Embed and store
An embedding model converts each text chunk into a numeric vector. Similar chunks produce similar vectors — this is what enables semantic search.

FAISS (Facebook AI Similarity Search) is an open-source library built by Meta's Fundamental AI Research (FAIR) group. It is specifically designed for efficient similarity search and clustering of dense vectors.

FAISS vs Chroma vs Pinecone: FAISS is great for local development (no server). Chroma is easy to set up with persistence. Pinecone / Weaviate / Qdrant are cloud-hosted for production scale.

In [ ]:
# Import modules for creating embeddings and storing them in a vector store
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Instantiate the embeddings model with the specified model name
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed all chunks and store in FAISS (in-memory, no server needed)
vector_store = FAISS.from_documents(chunks, embeddings)

# Save to disk so you don't re-embed every time
vector_store.save_local("faiss_index")

# Load later
vector_store = FAISS.load_local("faiss_index", embeddings,
                                allow_dangerous_deserialization=True)

#### Step 4 — Retrieve
A retriever takes a query string and returns the most relevant chunks. The default is similarity search (cosine distance between embeddings).

MMR — Maximum Marginal Relevance

Instead of returning the top-k most similar chunks (which may all say the same thing), MMR balances relevance with diversity. Use search_type="mmr" when your chunks have redundant content.

Multi-query retrieval

Generate multiple phrasings of the user's question and retrieve for each. Useful when the user's phrasing doesn't match how the document was written.

from langchain.retrievers import MultiQueryRetriever retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",  # or "mmr"
    search_kwargs={"k": 4},    # return top-4 chunks (Default: 4)
)

docs = retriever.invoke("Who are Mashhhood Raza and Aamna Hasan?")

#### Step 5 — 

Use ChatPromptTemplate to induce the context of the document in your query to LLM.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer using the context below. If the answer is not in the context, say "I don't know".
Context: {context}
Question: {question}""")

#### Step 6 — Answer with a RAG chain

Combine the retriever and LLM into an LCEL that fetches context and answers in one call.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Instantiate the ChatOpenAI model with the specified model name and temperature
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Chain without RAG
chain = prompt | model | StrOutputParser()
response = chain.invoke("Who are Mashhhood Raza and Aamna Hasan?")
print(f'Without RAG: {response}')

# Chain with RAG
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt 
    | model 
    | StrOutputParser()
)
rag_response = rag_chain.invoke("Who are Mashhhood Raza and Aamna Hasan?")
print(f'With RAG: {rag_response}')

KeyboardInterrupt: 